# Perfilado de datos (tarea 2.1)
Lee Files/raw de lh_gastos, calcula el perfil y guarda Files/profiling/perfilado.json.

In [ ]:
import json
from pyspark.sql import functions as F

WS = "2fab3dda-136a-4701-92f1-c98355abb356"; LH = "2813d425-4d72-4fc7-889e-168aef58cbbf"
B = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}/Files/raw"
OUT = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}/Files/profiling/perfilado.json"

def rd(p):
    return spark.read.option("header", True).option("inferSchema", True).csv(f"{B}/{p}")

T = {
    "articulos": rd("01_inicial/articulos.csv"),
    "familias": rd("01_inicial/familias.csv"),
    "operadores": rd("01_inicial/operadores.csv"),
    "sucursales": rd("01_inicial/sucursales.csv"),
    "terceros": rd("01_inicial/terceros.csv"),
    "presupuestos": rd("01_inicial/presupuestos.csv"),
    "documentos_inicial": rd("01_inicial/documentos.csv"),
    "detalle_inicial": rd("01_inicial/detalle_documentos.csv"),
    "documentos_incremental": rd("02_incremental/documentos.csv"),
    "detalle_incremental": rd("02_incremental/detalle_documentos.csv"),
}

def perfil(df):
    aggs = []
    for c, t in df.dtypes:
        aggs.append(F.sum(F.col(c).isNull().cast("int")).alias(c + "__nulos"))
        aggs.append(F.approx_count_distinct(c).alias(c + "__dist"))
        aggs.append(F.min(c).alias(c + "__min"))
        aggs.append(F.max(c).alias(c + "__max"))
    aggs.append(F.count(F.lit(1)).alias("__n"))
    r = df.agg(*aggs).first().asDict()
    cols = []
    for c, t in df.dtypes:
        cols.append({"columna": c, "tipo": t, "nulos": int(r[c + "__nulos"] or 0),
                     "distintos_aprox": int(r[c + "__dist"]), "min": str(r[c + "__min"]), "max": str(r[c + "__max"])})
    return {"filas": int(r["__n"]), "columnas": cols}

def dist(df, col):
    return {str(r[col]): int(r["count"]) for r in df.groupBy(col).count().orderBy(col).collect()}

R = {"tablas": {k: perfil(v) for k, v in T.items()}}

D = lambda c: F.col(c).cast("decimal(18,2)")

def chequeos_lote(doc, det, cat, nombre):
    out = {}
    out["estado"] = dist(doc, "estado")
    out["tipo"] = dist(doc, "tipo")
    out["documentos_duplicados_por_id"] = doc.count() - doc.select("documento_id").distinct().count()
    out["lineas_duplicadas_por_linea_id"] = det.count() - det.select("linea_id").distinct().count()
    out["lineas_por_documento"] = {str(r["count"]): int(r["n"]) for r in det.groupBy("documento_id").count().groupBy("count").agg(F.count(F.lit(1)).alias("n")).collect()}
    out["numero_linea"] = dist(det, "numero_linea")
    out["descuento_pct"] = dist(det, "descuento_pct")
    out["cantidad_min_max"] = [str(x) for x in det.agg(F.min("cantidad"), F.max("cantidad")).first()]
    out["huerfanos"] = {
        "detalle_sin_documento": det.join(doc, "documento_id", "left_anti").count(),
        "detalle_sin_articulo": det.join(cat["articulos"], "articulo_id", "left_anti").count(),
        "documento_sin_sucursal": doc.join(cat["sucursales"], "sucursal_id", "left_anti").count(),
        "documento_sin_operador": doc.join(cat["operadores"], "operador_id", "left_anti").count(),
        "documento_sin_tercero": doc.join(cat["terceros"], "tercero_id", "left_anti").count(),
        "documento_sin_detalle": doc.join(det.select("documento_id").distinct(), "documento_id", "left_anti").count(),
    }
    base = F.round((D("cantidad") * D("precio_unitario") * (F.lit(100) - F.col("descuento_pct").cast("decimal(18,2)")) / F.lit(100)).cast("decimal(30,6)"), 2)
    imp = F.round((D("base_neta") * F.lit(0.12)).cast("decimal(30,6)"), 2)
    chk = det.select(
        (base != D("base_neta")).cast("int").alias("e_base"),
        (imp != D("impuesto")).cast("int").alias("e_imp"),
        ((D("base_neta") + D("impuesto")) != D("total_linea")).cast("int").alias("e_tot"),
    ).agg(F.sum("e_base"), F.sum("e_imp"), F.sum("e_tot")).first()
    out["diferencias_calculo_lineas"] = {"base_neta": int(chk[0] or 0), "impuesto": int(chk[1] or 0), "total_linea": int(chk[2] or 0)}
    sumas = det.groupBy("documento_id").agg(F.sum(D("total_linea")).alias("s"))
    out["total_documento_distinto_de_suma_lineas"] = doc.join(sumas, "documento_id").where(D("total_documento") != F.col("s")).count()
    out["fecha"] = {"min": str(doc.agg(F.min("fecha_documento")).first()[0]), "max": str(doc.agg(F.max("fecha_documento")).first()[0])}
    out["fecha_actualizacion_distintas"] = [str(r[0]) for r in doc.select("fecha_actualizacion").distinct().collect()]
    out["vencimiento_anterior_a_fecha"] = doc.where(F.col("fecha_vencimiento") < F.col("fecha_documento")).count()
    mensual = doc.groupBy(F.date_format("fecha_documento", "yyyy-MM").alias("m")).count()
    out["meses"] = {"cantidad": mensual.count(), "min_docs": mensual.agg(F.min("count")).first()[0], "max_docs": mensual.agg(F.max("count")).first()[0]}
    vig = det.join(doc.where(F.col("estado") == "VIGENTE").select("documento_id"), "documento_id")
    out["base_neta_vigente_total"] = str(vig.agg(F.sum(D("base_neta"))).first()[0])
    out["base_neta_total_incluye_anulados"] = str(det.agg(F.sum(D("base_neta"))).first()[0])
    return out

cat = {"articulos": T["articulos"], "sucursales": T["sucursales"], "operadores": T["operadores"], "terceros": T["terceros"]}
R["lote_inicial"] = chequeos_lote(T["documentos_inicial"], T["detalle_inicial"], cat, "inicial")
R["lote_incremental"] = chequeos_lote(T["documentos_incremental"], T["detalle_incremental"], cat, "incremental")
R["traslape_inicial_vs_incremental"] = {
    "documento_id_en_ambos": T["documentos_inicial"].join(T["documentos_incremental"], "documento_id").count(),
    "linea_id_en_ambos": T["detalle_inicial"].join(T["detalle_incremental"], "linea_id").count(),
}

p = T["presupuestos"]
R["presupuestos"] = {
    "periodos": {"min": str(p.agg(F.min("periodo")).first()[0]), "max": str(p.agg(F.max("periodo")).first()[0]), "distintos": p.select("periodo").distinct().count()},
    "duplicados_clave_periodo_sucursal_articulo": p.count() - p.select("periodo", "sucursal_id", "articulo_id").distinct().count(),
    "sin_sucursal": p.join(T["sucursales"], "sucursal_id", "left_anti").count(),
    "sin_articulo": p.join(T["articulos"], "articulo_id", "left_anti").count(),
    "importe_min_max_total": [str(x) for x in p.agg(F.min("importe_presupuesto"), F.max("importe_presupuesto"), F.sum(D("importe_presupuesto"))).first()],
}
R["catalogos"] = {
    "sucursales_por_region": dist(T["sucursales"], "region"),
    "sucursales_por_departamento": dist(T["sucursales"], "departamento"),
    "terceros_por_segmento": dist(T["terceros"], "segmento"),
    "terceros_plazo_dias": dist(T["terceros"], "plazo_dias"),
    "operadores_por_cargo": dist(T["operadores"], "cargo"),
}

s = json.dumps(R, default=str, ensure_ascii=False, indent=1)
notebookutils.fs.put(OUT, s, True)
print(s)
